# XGBoost para Estimativa de Redshift de ELGs

Pipeline com **XGBoost** + **Optuna** (tuning de hiperparâmetros) para predição
de redshift de **Emission Line Galaxies (ELGs)** do eBOSS/SDSS DR17.

**Dataset:** `ELGspectra_padded.h5` — espectros 1D (~4635 pts).

> **Nota:** XGBoost é um modelo tabular. Os 4635 pontos do espectro são tratados
> como features. Para datasets muito grandes, considerar redução de
> dimensionalidade (PCA) ou extração de features (forças de linhas).

**Estrutura do notebook:**
1. Imports e configuração
2. Configurações do dataset
3. Carregar dados
4. Pré-processamento e split
5. Baseline XGBoost
6. Tuning de hiperparâmetros com Optuna
7. Treinar modelo final com best params
8. Avaliação e visualização
9. Análise de outliers
10. Salvar modelo


## 1. Imports e Configuração


In [ ]:
import time
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Reprodutibilidade
np.random.seed(42)
SEED = 42


## 2. Configurações do Dataset

Importa caminhos do `config.py` (detecta automaticamente cluster vs local).


In [ ]:
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, RESULTS_DIR as PROJECT_RESULTS, MODELS_DIR, print_info

print_info()

paths = paths_for("ELG")
HDF5_PATH   = paths["spectra_h5"].with_name("ELGspectra_padded.h5")
RESULTS_DIR = PROJECT_RESULTS / "ELG" / "xgboost"
MODEL_DIR   = MODELS_DIR / "ELG" / "xgboost"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nHDF5_PATH:   {HDF5_PATH}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"MODEL_DIR:   {MODEL_DIR}")


## 3. Carregar Dados

Lê o HDF5 com os espectros já com padding aplicado.
Use `n_samples=N` para testar com um subconjunto.


In [ ]:
def load_data(hdf5_path, n_samples=None):
    with h5py.File(hdf5_path, 'r') as f:
        total = f['ml_dataset/y'].shape[0]
        n_pts = f['ml_dataset/X_spec'].shape[1]

        if n_samples and n_samples < total:
            np.random.seed(SEED)
            indices = np.sort(np.random.choice(total, n_samples, replace=False))
            X = f['ml_dataset/X_spec'][indices]
            y = f['ml_dataset/y'][indices]
        else:
            X = f['ml_dataset/X_spec'][:]
            y = f['ml_dataset/y'][:]

    print(f"Espectros : {len(y):,}")
    print(f"Features  : {n_pts}")
    print(f"Redshift  : {y.min():.3f} - {y.max():.3f}")
    return X, y, n_pts


X, y, n_features = load_data(HDF5_PATH, n_samples=None)


## 4. Pré-processamento e Split

- **Normalização por espectro:** divide cada espectro pelo seu máximo absoluto
  (preserva forma das linhas, robusto a zeros do padding).
- **Split:** 70% train / 15% val (para Optuna) / 15% test


In [ ]:
def normalize_spectra(X_spec):
    """Divide cada espectro pelo seu max absoluto."""
    norms = np.abs(X_spec).max(axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return X_spec / norms


X_norm = normalize_spectra(X)

# Split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f"Train: {len(y_train):,}")
print(f"Val:   {len(y_val):,}")
print(f"Test:  {len(y_test):,}")


## 5. Baseline XGBoost

Modelo XGBoost com hiperparâmetros padrão para termos uma referência antes
do tuning. Use `tree_method='hist'` (rápido) ou `'gpu_hist'` (se tiver GPU).


In [ ]:
baseline_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',     # use 'gpu_hist' no cluster com GPU
    'device': 'cpu',           # mude para 'cuda' no cluster com GPU
    'n_estimators': 500,
    'learning_rate': 0.1,
    'max_depth': 6,
    'random_state': SEED,
    'n_jobs': -1,
}

start = time.time()
baseline = xgb.XGBRegressor(**baseline_params)
baseline.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
baseline_time = time.time() - start

y_pred_baseline = baseline.predict(X_test)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_baseline   = r2_score(y_test, y_pred_baseline)
print(f"Baseline:  RMSE={rmse_baseline:.4f}  R2={r2_baseline:.4f}  ({baseline_time:.0f}s)")


## 6. Tuning com Optuna

Otimização Bayesiana de hiperparâmetros usando o conjunto de validação como
sinal. O estudo é salvo em SQLite (`xgboost_study.db`), o que permite **pausar
e continuar** o tuning depois (basta rodar a célula de novo).

**Hiperparâmetros tunados:**
- `n_estimators`, `max_depth`, `learning_rate`
- `subsample`, `colsample_bytree`
- `min_child_weight`, `reg_alpha`, `reg_lambda`, `gamma`

**Custo:** cada trial treina um modelo do zero. Para datasets grandes,
considere reduzir `N_TRIALS` ou usar `max_depth` menor.


In [ ]:
N_TRIALS = 30   # aumente para tuning mais agressivo

def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cpu',
        'random_state': SEED,
        'n_jobs': -1,
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1500, step=100),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
    }

    model = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    y_pred_val = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    return rmse


# Estudo persistente: pode pausar e continuar depois
study_path = MODEL_DIR / "xgboost_study.db"
study = optuna.create_study(
    direction='minimize',
    study_name='xgboost_elg',
    storage=f'sqlite:///{study_path}',
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

start = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
tune_time = time.time() - start

print(f"\nTempo de tuning: {tune_time:.0f}s")
print(f"Trials ja rodados (total): {len(study.trials)}")
print(f"Melhor RMSE (val): {study.best_value:.4f}")
print(f"\nMelhores parametros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


### Visualizações do estudo Optuna

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histórico de otimização
trials_df = study.trials_dataframe()
sns.lineplot(data=trials_df, x='number', y='value', ax=axes[0], marker='o', alpha=0.6)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('RMSE (val)')
axes[0].set_title('Histórico do Optuna', fontweight='bold')
axes[0].axhline(study.best_value, color='red', ls='--', label=f'Best: {study.best_value:.4f}')
axes[0].legend()

# Importância dos hiperparâmetros
try:
    importances = optuna.importance.get_param_importances(study)
    df_imp = pd.DataFrame({
        'param': list(importances.keys()),
        'importance': list(importances.values()),
    }).sort_values('importance', ascending=True)
    sns.barplot(data=df_imp, y='param', x='importance', ax=axes[1], color='steelblue')
    axes[1].set_title('Importância dos Hiperparâmetros', fontweight='bold')
except Exception as e:
    axes[1].text(0.5, 0.5, f'Importance falhou:\n{e}',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'optuna_study.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Treinar Modelo Final

Treina com os melhores hiperparâmetros encontrados pelo Optuna, usando o
conjunto de treino + validação combinados (mais dados para o modelo final).


In [ ]:
best_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cpu',
    'random_state': SEED,
    'n_jobs': -1,
    **study.best_params,
}

# Combinar train + val para o modelo final
X_trainval = np.concatenate([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

start = time.time()
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_trainval, y_trainval, verbose=False)
final_time = time.time() - start

print(f"Tempo de treino final: {final_time:.0f}s")


## 8. Avaliação no Teste

Métricas idênticas às da CNN para comparação direta.


In [ ]:
y_pred = final_model.predict(X_test)

rmse     = np.sqrt(mean_squared_error(y_test, y_pred))
mae      = mean_absolute_error(y_test, y_pred)
r2       = r2_score(y_test, y_pred)
delta_z  = (y_pred - y_test) / (1 + y_test)
bias     = np.median(delta_z)
nmad     = 1.48 * np.median(np.abs(delta_z - bias))
outliers = np.sum(np.abs(delta_z) > 0.01) / len(delta_z) * 100

results = {
    'rmse': rmse, 'mae': mae, 'r2': r2,
    'bias': bias, 'nmad': nmad, 'outliers': outliers,
    'y_test': y_test, 'y_pred': y_pred, 'delta_z': delta_z,
}

print(f"Resultados no teste:")
print(f"  RMSE:       {rmse:.4f}")
print(f"  MAE:        {mae:.4f}")
print(f"  R2:         {r2:.4f}")
print(f"  <delta_z>:  {bias:.4f}")
print(f"  sigma_NMAD: {nmad:.4f}")
print(f"  Outliers:   {outliers:.1f}%")
print(f"\nVs Baseline:  RMSE {rmse_baseline:.4f} -> {rmse:.4f}  (R2 {r2_baseline:.4f} -> {r2:.4f})")


### Visualização — Predito vs Real e Resíduos

In [ ]:
sns.set_theme(style='whitegrid', palette='deep')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('XGBoost — Resultados', fontsize=13, fontweight='bold')

# Predito vs real
ax = axes[0]
scatter = ax.scatter(y_test, y_pred, c=y_test, cmap='viridis', alpha=0.5, s=10)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('True Redshift')
ax.set_ylabel('Predicted Redshift')
ax.set_title(f'Predito vs Real (R2={r2:.4f}  RMSE={rmse:.4f})', fontweight='bold')
plt.colorbar(scatter, ax=ax, label='True Z')

# Histograma de delta_z
ax = axes[1]
sns.histplot(delta_z, bins=80, ax=ax, color='steelblue', alpha=0.7)
ax.axvline(0, color='red', ls='--', lw=1.5)
ax.axvline(bias, color='black', ls='--', lw=1.5, label=f'bias={bias:.4f}')
ax.set_xlabel('delta_z / (1+z)')
ax.set_ylabel('Contagem')
ax.set_title(f'Distribuicao de delta_z  (sigma_NMAD={nmad:.4f})', fontweight='bold')
ax.legend()

# Importância das features (top 30)
ax = axes[2]
importances = final_model.feature_importances_
top_n = 30
top_idx = np.argsort(importances)[-top_n:]
ax.barh(np.arange(top_n), importances[top_idx], color='steelblue')
ax.set_yticks(np.arange(top_n))
ax.set_yticklabels([f'pix {i}' for i in top_idx], fontsize=7)
ax.set_xlabel('Importância')
ax.set_title(f'Top {top_n} features (pixels do espectro)', fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgboost_results.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Análise de Outliers

Mesmo critério usado na CNN: `|Δz/(1+z)| > 0.01`.


In [ ]:
threshold = 0.01
outlier_mask = np.abs(delta_z) > threshold
inlier_mask  = ~outlier_mask
n_out, n_tot = outlier_mask.sum(), len(delta_z)

print(f"Outliers totais: {n_out} / {n_tot}  ({100*n_out/n_tot:.1f}%)")
print()

for label, mask in [("Inliers", inlier_mask), ("Outliers", outlier_mask)]:
    if mask.sum() == 0:
        continue
    yt, yp, dz = y_test[mask], y_pred[mask], delta_z[mask]
    rmse_  = np.sqrt(mean_squared_error(yt, yp))
    mae_   = mean_absolute_error(yt, yp)
    r2_    = r2_score(yt, yp) if mask.sum() > 1 else float('nan')
    bias_  = np.median(dz)
    nmad_  = 1.48 * np.median(np.abs(dz - bias_))
    print(f"  {label} (n={mask.sum():,}):")
    print(f"    RMSE={rmse_:.4f}  MAE={mae_:.4f}  R2={r2_:.4f}")
    print(f"    <delta_z>={bias_:.4f}  sigma_NMAD={nmad_:.4f}")
    print(f"    z range: [{yt.min():.3f}, {yt.max():.3f}]")
    print()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'XGBoost - Analise de Outliers (|delta_z/(1+z)| > {threshold})',
             fontsize=12, fontweight='bold')

ax = axes[0]
sns.scatterplot(x=y_test[inlier_mask], y=y_pred[inlier_mask],
                s=30, alpha=0.4, color='steelblue', label='Inliers', ax=ax)
sns.scatterplot(x=y_test[outlier_mask], y=y_pred[outlier_mask],
                s=60, alpha=0.7, color='red', marker='x', label='Outliers', ax=ax)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=1.5)
ax.set_xlabel('True Redshift')
ax.set_ylabel('Predicted Redshift')
ax.set_title('Predito vs Real', fontweight='bold')

ax = axes[1]
df_hist = pd.DataFrame({
    'delta_z/(1+z)': np.concatenate([delta_z[inlier_mask], delta_z[outlier_mask]]),
    'Tipo': ['Inliers']*inlier_mask.sum() + ['Outliers']*outlier_mask.sum()
})
sns.histplot(data=df_hist, x='delta_z/(1+z)', hue='Tipo', bins=60, alpha=0.6,
             ax=ax, palette=['steelblue', 'red'])
ax.axvline(threshold,  color='k', ls='--', lw=1.5)
ax.axvline(-threshold, color='k', ls='--', lw=1.5)
ax.set_xlabel('delta_z / (1+z)')
ax.set_ylabel('Contagem')
ax.set_title('Distribuicao de delta_z', fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgboost_outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Salvar Modelo e Métricas

Salva o modelo treinado e um CSV com as métricas para ser carregado pelo
notebook de comparação ([04_comparison_elg.ipynb](04_comparison_elg.ipynb)).


In [ ]:
# Modelo
model_path = MODEL_DIR / 'xgboost_elg_final.json'
final_model.save_model(model_path)
print(f"Modelo salvo: {model_path}")

# Predições e métricas
np.savez(
    RESULTS_DIR / 'xgboost_predictions.npz',
    y_test=y_test, y_pred=y_pred, delta_z=delta_z,
)

metrics_df = pd.DataFrame([{
    'model': 'xgboost',
    'rmse': rmse, 'mae': mae, 'r2': r2,
    'bias': bias, 'nmad': nmad, 'outliers_pct': outliers,
}])
metrics_df.to_csv(RESULTS_DIR / 'xgboost_metrics.csv', index=False)
print(f"Metricas salvas:  {RESULTS_DIR / 'xgboost_metrics.csv'}")
print(f"Predicoes salvas: {RESULTS_DIR / 'xgboost_predictions.npz'}")
